# Myeloid annotation 

**Gene list sources**  
[Aegerter et al. 2022, Cell Immunity](https://www.sciencedirect.com/science/article/pii/S1074761322004034?via%3Dihub)  
[Liu et al. 2022, Cell Immunity](https://www.sciencedirect.com/science/article/pii/S1074761321003447)  
[Singh et al. 2016, Nature Communications](https://www.nature.com/articles/ncomms13581)

## Python modules

In [ ]:
import scanpy as sc
import anndata as ad

import scanorama
import scvi

from scipy import sparse
import pandas as pd
import numpy as np

import pickle

import seaborn as sb
import matplotlib.pyplot as plt

import os

## Settup rpy2 

In [ ]:
os.environ['R_HOME'] = '/nobackup/peer/fdeckert/miniconda3/envs/r.4.1.0-FD20220623HSCT/lib/R'

In [ ]:
import rpy2.rinterface_lib.callbacks
import logging

from rpy2.robjects import pandas2ri
import anndata2ri

In [ ]:
%load_ext rpy2.ipython

## Settings

In [ ]:
# Working dir
os.chdir('/research/peer/fdeckert/FD20220623HSCT')

In [ ]:
# Warnings Python 
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Warnings R
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR) 

## R modules

In [76]:
%%R
library(Seurat)
library(dplyr)

## Plotting 

In [ ]:
# Scanpy figures
sc.set_figure_params(figsize=(5, 5), dpi_save=1200, fontsize=12, frameon=False, color_map='viridis', facecolor="white")

In [ ]:
# Matplot figures
plt.rcParams['figure.figsize']=(7, 7)

In [ ]:
# Load color 
import rpy2.robjects as robjects
color_load = robjects.r.source('plotting_global.R')
color = dict()
for i in range(len(color_load[0])):
    color[color_load[0].names[i]] = {key : color_load[0][i].rx2(key)[0] for key in color_load[0][i].names}

In [ ]:
# Set color function 
def set_color(adata, categories): 
    
    categories = [x for x in categories if x in list(adata.obs.columns)]

    for category in categories: 

        adata.obs[category] = pd.Series(adata.obs[category], dtype='category').cat.remove_unused_categories()
        
        keys = list(color[category].keys())
        keys = [x for x in keys if x in list(adata.obs[category])]        

        adata.obs[category] = adata.obs[category].cat.reorder_categories(keys)
        adata.uns[category+'_colors'] = np.array([color[category].get(key) for key in keys], dtype=object)
        
    return(adata)

# Processing functions 

## Dim reduction and cluster function 

In [ ]:
def dim_clust(adata, use_rep=None, n_neighbors=15, resolution=1, min_dist=0.5, metric='euclidean', file=None): 
    
    # Dim reduction and clustering on the scanorama corrected embedding 
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep=use_rep, metric=metric)
    if resolution is not None: 
        sc.tl.leiden(adata, resolution=resolution)
    
    sc.tl.umap(adata, min_dist=min_dist)
    
    # Write output 
    if file is not None: 
        adata.write_h5ad(file)
    
    return(adata)

## Differential expression analysis 

In [ ]:
def DEA(adata, key): 
    
    # Normalize
    sc.pp.normalize_total(adata, target_sum=1e6)
    sc.pp.log1p(adata)
    
    # DEA Wilcoxon 
    sc.tl.rank_genes_groups(adata, key, method='wilcoxon', key_added="wilcoxon", use_raw=False)
    sc.pl.rank_genes_groups(adata, n_genes=25, fontsize=12, sharey=False, key="wilcoxon", ncols=5)

# Annotate blood subset 

## Import data

In [ ]:
adata = sc.read_h5ad('data/object/subsets/round_3/myeloid_blood/scvi_con.h5ad')

In [ ]:
adata_raw = adata.raw.to_adata()
adata = adata.raw.to_adata()
adata.raw = adata_raw

In [ ]:
# Filter celltypist annotation 
adata.obs['celltypist_immun_low'] = adata.obs['celltypist_immun_low'].to_list()
adata.obs = adata.obs.apply(lambda x: x.mask(x.map(x.value_counts())<int(adata.n_obs/100), 'freq < 1 %') if x.name=='celltypist_immun_low' else x)

adata.obs['celltypist_immun_high'] = adata.obs['celltypist_immun_high'].to_list()
adata.obs = adata.obs.apply(lambda x: x.mask(x.map(x.value_counts())<int(adata.n_obs/100), 'freq < 1 %') if x.name=='celltypist_immun_high' else x)

## Gene modules

In [ ]:
cell_type_genes_dict = {
    
    r'Monocytes': ['CD14', 'FCGR3A', 'CCR2'], 
    r'Macrophages': ['CD163', 'MRC1', 'F13A1'], 
    r'cDC1': ['XCR1', 'THBD', 'CLEC9A'], 
    r'cDC2': ['CD1C', 'SIRPA', 'CLEC10A'], 
    r'LC': ['CD1A', 'CD207'], 
    r'pDC': ['IL3RA', 'TCF4', 'CLEC4C'], 
    r'Integrins': ['ITGAM', 'ITGAX', 'ITGAE'], 
    r'Activation': ['CD80', 'CD86', 'STAT1', 'FOS', 'JUN'], 
    r'Migration': ['CCR7'], 
    r'Maturation': ['LAMP3', 'BIRC3'], 
    r'Immune signaling': ['IL1A', 'IL1B', 'IL6', 'CXCL10'], 
    r'Stress response': ['HSPA8'], 
    r'Antigen presentation': ['HLA-A', 'HLA-B', 'HLA-C', 'HLA-DRA']
    
}

## Dim reduction 

In [ ]:
adata = dim_clust(adata, use_rep='X_scvi', n_neighbors=30, resolution=1, min_dist=0.5, metric='euclidean', file='data/object/subsets/annotation/myeloid_blood.h5ad')

In [ ]:
sc.pl.umap(adata, color=['leiden', 'cell_type_leiden', 'cc_phase_class', 'hto_demux_class', 'time_point', 'genotype_class', 'patient_id', 'has_ir', 'cc_aa_alignment_size', 'celltypist_immun_high', 'celltypist_immun_low', 'msS_RNA'], legend_loc='right margin', ncols=4, wspace=0.3, size=50)

## Fine Leiden annotation 

In [ ]:
leiden_cell_type_fine = {
    
    '0': r'$Mono_{nonCl}\ (0)$',
    '1': r'$Mono_{Cl}\ (1)$',
    '2': r'$Mono_{Cl}\ (2)$',
    '3': r'$Mono_{Cl}\ (3)$',
    '4': r'$Mono_{Cl}\ (4)$',
    '5': r'$Mono_{Cl}\ (5)$',
    '6': r'$Mono_{Cl}\ (6)$',
    '7': r'$Mono_{Int}\ (7)$',
    '8': r'$Mono_{Cl}\ (8)$',
    '9': r'$cDC2\ (9)$',
    '10': r'$Mono_{Cl}\ (10)$',
    '11': r'$pDC\ (11)$'    

}

In [ ]:
# Mapper 
leiden_cell_type_fine = pd.DataFrame({'leiden_cell_type_fine': adata.obs['leiden'].map(lambda x: leiden_cell_type_fine.get(x, x)).astype('category')})
adata.obs = adata.obs.merge(leiden_cell_type_fine, left_index=True, right_index=True, how='left')

In [ ]:
adata.obs['leiden_cell_type_fine'].cat.reorder_categories([
    
    
    r'$Mono_{Cl}\ (1)$',
    r'$Mono_{Cl}\ (2)$',
    r'$Mono_{Cl}\ (3)$',
    r'$Mono_{Cl}\ (4)$',
    r'$Mono_{Cl}\ (5)$',
    r'$Mono_{Cl}\ (6)$',
    r'$Mono_{Cl}\ (8)$',
    r'$Mono_{Cl}\ (10)$',
    r'$Mono_{Int}\ (7)$',
    r'$Mono_{nonCl}\ (0)$',
    r'$cDC2\ (9)$',
    r'$pDC\ (11)$' 
    
], inplace = True)

In [ ]:
dp = sc.pl.dotplot(adata, cell_type_genes_dict, groupby='leiden_cell_type_fine', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
DEA(adata.copy(), key='leiden_cell_type_fine')

## Main Leiden annotation 

In [ ]:
leiden_cell_type_main = {
    
    '0': r'$Mono_{nonCl}$',
    '1': r'$Mono_{Cl}$',
    '2': r'$Mono_{Cl}$',
    '3': r'$Mono_{Cl}$',
    '4': r'$Mono_{Cl}$',
    '5': r'$Mono_{Cl}$',
    '6': r'$Mono_{Cl}$',
    '7': r'$Mono_{Int}$',
    '8': r'$Mono_{Cl}$',
    '9': r'$cDC2$',
    '10': r'$Mono_{Cl}$',
    '11': r'$pDC$'  

}

In [ ]:
# Mapper 
leiden_cell_type_main = pd.DataFrame({'leiden_cell_type_main': adata.obs['leiden'].map(lambda x: leiden_cell_type_main.get(x, x)).astype('category')})
adata.obs = adata.obs.merge(leiden_cell_type_main, left_index=True, right_index=True, how='left')

In [ ]:
adata.obs['leiden_cell_type_main'].cat.reorder_categories([
    
    r'$Mono_{Cl}$', 
    r'$Mono_{Int}$', 
    r'$Mono_{nonCl}$',
    r'$cDC2$', 
    r'$pDC$'
    
], inplace = True)

In [ ]:
dp = sc.pl.dotplot(adata, cell_type_genes_dict, groupby='leiden_cell_type_main', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
DEA(adata.copy(), key='leiden_cell_type_main')

## Store annotation by cell id 

In [ ]:
leiden_cell_type_main_blood = adata.obs[['leiden', 'leiden_cell_type_main']]

# Annotate skin subset 

## Import data

In [ ]:
adata = sc.read_h5ad('data/object/subsets/round_3/myeloid_skin/scvi_con.h5ad')

In [ ]:
adata_raw = adata.raw.to_adata()
adata = adata.raw.to_adata()
adata.raw = adata_raw

In [ ]:
# Filter celltypist annotation 
adata.obs['celltypist_immun_low'] = adata.obs['celltypist_immun_low'].to_list()
adata.obs = adata.obs.apply(lambda x: x.mask(x.map(x.value_counts())<int(adata.n_obs/100), 'freq < 1 %') if x.name=='celltypist_immun_low' else x)

adata.obs['celltypist_immun_high'] = adata.obs['celltypist_immun_high'].to_list()
adata.obs = adata.obs.apply(lambda x: x.mask(x.map(x.value_counts())<int(adata.n_obs/100), 'freq < 1 %') if x.name=='celltypist_immun_high' else x)

## Gene modules

In [ ]:
cell_type_genes_dict = {
    
    r'Monocytes': ['CD14', 'FCGR3A', 'CCR2'], 
    r'Macrophages': ['CD163', 'MRC1', 'F13A1'], 
    r'cDC1': ['XCR1', 'THBD', 'CLEC9A'], 
    r'cDC2': ['CD1C', 'SIRPA', 'CLEC10A'], 
    r'LC': ['CD1A', 'CD207'], 
    r'pDC': ['IL3RA', 'TCF4', 'CLEC4C'], 
    r'Integrins': ['ITGAM', 'ITGAX', 'ITGAE'], 
    r'Activation': ['CD80', 'CD86', 'STAT1', 'FOS', 'JUN'], 
    r'Migration': ['CCR7'], 
    r'Maturation': ['LAMP3', 'BIRC3'], 
    r'Immune signaling': ['IL1A', 'IL1B', 'IL6', 'CXCL10'], 
    r'Stress response': ['HSPA8'], 
    r'Antigen presentation': ['HLA-A', 'HLA-B', 'HLA-C', 'HLA-DRA']
    
}

## Dim reduction 

In [ ]:
adata = dim_clust(adata, use_rep='X_scvi', n_neighbors=30, resolution=1, min_dist=0.5, metric='euclidean', file='data/object/subsets/annotation/myeloid_skin.h5ad')

In [ ]:
sc.pl.umap(adata, color=['leiden', 'cell_type_leiden', 'cc_phase_class', 'hto_demux_class', 'time_point', 'genotype_class', 'patient_id', 'has_ir', 'cc_aa_alignment_size', 'celltypist_immun_high', 'celltypist_immun_low', 'msS_RNA'], legend_loc='right margin', ncols=4, wspace=0.3, size=50)

## Fine Leiden annotation 

In [ ]:
leiden_cell_type_fine = {
    
    '0': r'$M\phi_{Act}\ (0)$',
    '1': r'$M\phi\ (1)$',
    '2': r'$M\phi_{Inf}\ (2)$',
    '3': r'$cDC2\ (3)$',
    '4': r'$M\phi_{Inf}\ (4)$',
    '5': r'$M\phi_{Act}\ (5)$',
    '6': r'$cDC2\ (6)$',
    '7': r'$M\phi\ (7)$',
    '8': r'$M\phi\ (8)$',
    '9': r'$cDC1\ (9)$',
    '10': r'$pDC\ (10)$',
    '11': r'$LC\ (11)$',
    '12': r'$cDC_{Mig}\ (12)$'

}

In [ ]:
# Mapper 
leiden_cell_type_fine = pd.DataFrame({'leiden_cell_type_fine': adata.obs['leiden'].map(lambda x: leiden_cell_type_fine.get(x, x)).astype('category')})
adata.obs = adata.obs.merge(leiden_cell_type_fine, left_index=True, right_index=True, how='left')

In [ ]:
adata.obs['leiden_cell_type_fine'].cat.reorder_categories([
    
    r'$M\phi\ (1)$',
    r'$M\phi\ (7)$',
    r'$M\phi\ (8)$',
    r'$M\phi_{Act}\ (0)$',
    r'$M\phi_{Act}\ (5)$',
    r'$M\phi_{Inf}\ (2)$',
    r'$M\phi_{Inf}\ (4)$',
    r'$cDC1\ (9)$',
    r'$cDC2\ (3)$',
    r'$cDC2\ (6)$',
    r'$LC\ (11)$',
    r'$cDC_{Mig}\ (12)$', 
    r'$pDC\ (10)$'

    
], inplace = True)

In [ ]:
dp = sc.pl.dotplot(adata, cell_type_genes_dict, groupby='leiden_cell_type_fine', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
DEA(adata.copy(), key='leiden_cell_type_fine')

## Main Leiden annotation 

In [ ]:
leiden_cell_type_main = {
    
    '0': r'$M\phi_{Act}$',
    '1': r'$M\phi$',
    '2': r'$M\phi_{Inf}$',
    '3': r'$cDC2$',
    '4': r'$M\phi_{Inf}$',
    '5': r'$M\phi_{Act}$',
    '6': r'$cDC2$',
    '7': r'$M\phi$',
    '8': r'$M\phi$',
    '9': r'$cDC1$',
    '10': r'$pDC$',
    '11': r'$LC$',
    '12': r'$cDC_{Mig}$'

}

In [ ]:
# Mapper 
leiden_cell_type_main = pd.DataFrame({'leiden_cell_type_main': adata.obs['leiden'].map(lambda x: leiden_cell_type_main.get(x, x)).astype('category')})
adata.obs = adata.obs.merge(leiden_cell_type_main, left_index=True, right_index=True, how='left')

In [ ]:
adata.obs['leiden_cell_type_main'].cat.reorder_categories([
    
    r'$M\phi$',
    r'$M\phi_{Act}$',
    r'$M\phi_{Inf}$',
    r'$cDC1$',
    r'$cDC2$',
    r'$LC$',
    r'$cDC_{Mig}$', 
    r'$pDC$'
    
], inplace = True)

In [ ]:
dp = sc.pl.dotplot(adata, cell_type_genes_dict, groupby='leiden_cell_type_main', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
DEA(adata.copy(), key='leiden_cell_type_main')

## Store annotation by cell id 

In [ ]:
leiden_cell_type_main_skin = adata.obs[['leiden', 'leiden_cell_type_main']]

# Combine 

In [ ]:
adata = sc.read_h5ad('data/object/subsets/round_3/myeloid/scvi_con.h5ad')

In [ ]:
adata_raw = adata.raw.to_adata()
adata = adata.raw.to_adata()
adata.raw = adata_raw

In [ ]:
# Filter celltypist annotation 
adata.obs['celltypist_immun_low'] = adata.obs['celltypist_immun_low'].to_list()
adata.obs = adata.obs.apply(lambda x: x.mask(x.map(x.value_counts())<int(adata.n_obs/100), 'freq < 1 %') if x.name=='celltypist_immun_low' else x)

adata.obs['celltypist_immun_high'] = adata.obs['celltypist_immun_high'].to_list()
adata.obs = adata.obs.apply(lambda x: x.mask(x.map(x.value_counts())<int(adata.n_obs/100), 'freq < 1 %') if x.name=='celltypist_immun_high' else x)

## Gene modules

In [ ]:
cell_type_genes_dict = {
    
    r'Monocytes': ['CD14', 'FCGR3A', 'CCR2'], 
    r'Macrophages': ['CD163', 'MRC1', 'F13A1'], 
    r'cDC1': ['XCR1', 'THBD', 'CLEC9A'], 
    r'cDC2': ['CD1C', 'SIRPA', 'CLEC10A'], 
    r'LC': ['CD1A', 'CD207'], 
    r'pDC': ['IL3RA', 'TCF4', 'CLEC4C'], 
    r'Integrins': ['ITGAM', 'ITGAX', 'ITGAE'], 
    r'Activation': ['CD80', 'CD86', 'STAT1', 'FOS', 'JUN'], 
    r'Migration': ['CCR7'], 
    r'Maturation': ['LAMP3', 'BIRC3'], 
    r'Immune signaling': ['IL1A', 'IL1B', 'IL6', 'CXCL10'], 
    r'Stress response': ['HSPA8'], 
    r'Antigen presentation': ['HLA-A', 'HLA-B', 'HLA-C', 'HLA-DRA']
    
}

## Add combined blood and skin annotation 

In [ ]:
leiden_cell_type_main = pd.concat([leiden_cell_type_main_blood, leiden_cell_type_main_skin])
leiden_cell_type_main['leiden_cell_type_main'] = leiden_cell_type_main['leiden_cell_type_main'].astype('category')

del adata.obs['leiden']
adata.obs = adata.obs.merge(leiden_cell_type_main, left_index=True, right_index=True, how='left')

In [ ]:
adata.obs['leiden_cell_type_main'].cat.reorder_categories([
    
    r'$Mono_{Cl}$', 
    r'$Mono_{Int}$', 
    r'$Mono_{nonCl}$',
    r'$M\phi$',
    r'$M\phi_{Act}$',
    r'$M\phi_{Inf}$',
    r'$cDC1$',
    r'$cDC2$',
    r'$LC$',
    r'$cDC_{Mig}$', 
    r'$pDC$'
    
    
], inplace = True)

## Dim reduction 

In [ ]:
adata = dim_clust(adata, use_rep='X_scvi', n_neighbors=30, resolution=None, min_dist=0.5, metric='euclidean', file='data/object/subsets/annotation/myeloid.h5ad')

In [ ]:
sc.pl.umap(adata, color=['leiden', 'leiden_cell_type_main', 'cc_phase_class', 'hto_demux_class', 'time_point', 'genotype_class', 'patient_id', 'has_ir', 'cc_aa_alignment_size', 'celltypist_immun_high', 'celltypist_immun_low', 'msS_RNA'], legend_loc='right margin', ncols=4, wspace=0.3, size=50)

In [ ]:
dp = sc.pl.dotplot(adata, cell_type_genes_dict, groupby='leiden_cell_type_main', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

## Set colors

In [ ]:
adata = set_color(adata, list(color.keys()))

In [ ]:
sc.pl.umap(adata, color=['leiden_cell_type_main'], legend_loc='right margin', ncols=4, wspace=0.3, size=50)

In [ ]:
adata

# Save result CSV

In [78]:
adata.obs.loc[:, ['leiden_cell_type_main']].rename(columns={'leiden_cell_type_main':'cell_type_leiden_subset'}).to_csv('result/annotation/sbusets/myeloid_p.csv')

# Save AnnData object

In [ ]:
adata.write_h5ad('data/object/subsets/annotation/myeloid.h5ad')

# Save Seurat object

In [71]:
adata_r = adata.copy()

In [72]:
label_format = {
    
    r'$Mono_{Cl}$': "Mono['Cl']", 
    r'$Mono_{Int}$': "Mono['Int']", 
    r'$Mono_{nonCl}$': "Mono['nonCl']",
    r'$M\phi$': "M*Phi",
    r'$M\phi_{Act}$': "M*Phi['Act']",
    r'$M\phi_{Inf}$': "M*Phi['Inf']",
    r'$cDC1$': "cDC1",
    r'$cDC2$': "cDC2",
    r'$LC$': "LC",
    r'$cDC_{Mig}$': "cDC['Mig']", 
    r'$pDC$': "pDC"
    
}

adata_r.obs['leiden_cell_type_main'] = adata_r.obs['leiden_cell_type_main'].map(lambda x: label_format.get(x, x)).astype('category')

In [73]:
label_format_order = [
    
    "Mono['Cl']", 
    "Mono['Int']", 
    "Mono['nonCl']",
    "M*Phi",
    "M*Phi['Act']",
    "M*Phi['Inf']",
    "cDC1",
    "cDC2",
    "LC",
    "cDC['Mig']", 
    "pDC"
    
]

In [74]:
cnt = adata_r.X.T.todense()
cell_id = adata_r.obs_names
gene_id = adata_r.var_names
meta = adata_r.obs
umap = adata_r.obsm['X_umap']

In [77]:
%%R -i cnt -i cell_id -i gene_id -i meta -i label_format_order -i umap

colnames(cnt) <- cell_id
rownames(cnt) <- gene_id

so <- Seurat::CreateSeuratObject(cnt, meta.data=meta, project="lymphocyte", assay="RNA")

colnames(umap) <- c("UMAP_1", "UMAP_2")
rownames(umap) <- cell_id
so[["umap"]] <- CreateDimReducObject(embeddings=as.matrix(umap), key="UMAP_")

so$leiden_cell_type_main <- factor(so$leiden_cell_type_main, levels=label_format_order)
so$leiden_cell_type_fine <- factor(paste(so$leiden_cell_type_main, paste0("*' (", so$leiden, ")'")))

write.csv(so@meta.data[, "leiden_cell_type_main", drop=FALSE] %>% dplyr::rename(cell_type_leiden_subset="leiden_cell_type_main"), "result/annotation/sbusets/myeloid_r.csv")

saveRDS(so, 'data/object/subsets/annotation/myeloid.rds')